In [2]:
import requests
import pandas as pd
from datetime import datetime
import pyarrow

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",
    "minmagnitude": 4,
    "latitude": 34.23,    # Latitude of the center point
    "longitude": 137.3,   # Longitude of the center point
    "maxradiuskm": 700,
    "starttime": '2025-06-01',
    "orderby": "time",
}

response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    eqs = data["features"]
    df = pd.json_normalize(eqs)
    df['created_at'] = pd.to_datetime(df['properties.time'], unit="ms").dt.strftime("%Y-%m-%d")
    df['updated_at'] = datetime.now().strftime("%Y-%m-%d")
    #print(df.shape)

    csv_path = 'tmp/api_earthq_fut3r.csv'
    parq_path = 'tmp/api_earthq_fut3r.parquet'
    df.to_csv(csv_path, index=False)
    df.to_parquet(parq_path, index=False)

else:
    print(response.status_code)

In [5]:
!pip install -U "pandas==2.1.4" "pyarrow==14.0.2" --break-system-packages